In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import time

class AdvancedMovementDetector:
    def __init__(self):
        self.frame_history = []
        self.camera_motion_history = []
        self.track_data_store = {}
        self.track_history_store = {}
        self.track_feature_store = {}  # NEW: Store appearance features
        
        # Configuration
        self.max_history = 15
        self.min_track_age = 2  # CHANGED: Reduced from 3 to 2
        
        # Distance sensitivity parameters - IMPROVED
        self.distance_sensitivity = 7.0  # CHANGED: Increased from 5.0 to 7.0
        self.distant_object_boost = 0.4  # CHANGED: Increased from 0.3 to 0.4
        self.far_threshold = 0.6
        
        # Front/rear motion detection parameters
        self.headlight_brightness_threshold = 500
        self.intersection_threshold_factor = 0.4  # CHANGED: Reduced from 0.5 to 0.4
        
        # Camera motion compensation parameters - IMPROVED
        self.camera_motion_area_factor = 0.007  # CHANGED: Reduced from 0.01 to 0.007
        self.camera_motion_threshold = 0.4     # CHANGED: Reduced from 0.5 to 0.4
        
        # NEW: Small movement boost parameters
        self.small_movement_boost_threshold = 6.0
        self.small_movement_boost_factor = 0.25
        
        # Debug mode to print detailed information
        self.debug = True
        
        print("Initialized Enhanced Movement Detector with improved tracking and motion detection")
    
    def process_frame(self, current_frame, previous_frame, detected_boxes, tracks):
        """Process a frame to detect moving objects with advanced techniques."""
        frame_h, frame_w = current_frame.shape[:2]
        frame_shape = (frame_h, frame_w)
        timestamp = time.time()
        
        # Store frames in history
        if len(self.frame_history) >= self.max_history:
            self.frame_history.pop(0)
        self.frame_history.append(current_frame.copy())
        
        # 1. Estimate camera motion using homography
        H, dx, dy, motion_data = self.estimate_camera_motion_homography(
            previous_frame, current_frame, detected_boxes
        )
        
        # Store camera motion in history
        camera_motion = {
            'homography': H,
            'dx': dx,
            'dy': dy,
            'data': motion_data,
            'timestamp': timestamp
        }
        
        if len(self.camera_motion_history) >= self.max_history:
            self.camera_motion_history.pop(0)
        self.camera_motion_history.append(camera_motion)
        
        # Extract camera motion magnitude for later use
        camera_mag = 0
        if camera_motion and 'data' in camera_motion and camera_motion['data']:
            camera_mag = camera_motion['data'].get('magnitude', 0)
        
        # Process each track
        results = []
        for track in tracks:
            if not track.is_confirmed():
                continue
                
            tid = track.track_id
            bbox = track.to_ltrb()
            x1, y1, x2, y2 = map(int, bbox)
            
            # Calculate size metrics
            obj_width = x2 - x1
            obj_height = y2 - y1
            obj_area = obj_width * obj_height
            size_ratio = obj_area / (frame_w * frame_h)
            
            # Calculate center point
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            
            # Check if vehicle is in an intersection
            is_in_intersection = (cx > frame_w*0.4 and cx < frame_w*0.6)
            
            # Check for headlights/taillights (brightness in lower half of object)
            is_front_facing = False
            if 0 <= y1 < y2 and 0 <= x1 < x2 and y2 <= frame_h and x2 <= frame_w:
                # Extract lower half of object
                lower_half_y1 = y1 + (y2 - y1) // 2
                lower_half = current_frame[lower_half_y1:y2, x1:x2]
                
                if lower_half.size > 0:  # Ensure we have valid image data
                    # Convert to grayscale and check brightness
                    if len(lower_half.shape) == 3:  # Color image
                        lower_half_gray = cv2.cvtColor(lower_half, cv2.COLOR_BGR2GRAY)
                        max_brightness = np.max(lower_half_gray)
                        
                        # Check if we have bright spots (headlights/taillights)
                        if max_brightness > self.headlight_brightness_threshold:
                            is_front_facing = True
            
            # NEW: Store trail position for visualization
            if tid not in self.track_data_store:
                self.track_data_store[tid] = {
                    'bbox_prev': bbox,
                    'patch_prev': None,
                    'age': 0,
                    'size_ratio': size_ratio,
                    'is_moving': False,
                    'consecutive_moving': 0,
                    'consecutive_static': 0,
                    'is_front_facing': is_front_facing,
                    'is_in_intersection': is_in_intersection,
                    'trail': [(cx, cy)]  # Initialize trail with current position
                }
                
                # Extract image patch (if valid coordinates)
                if 0 <= y1 < y2 and 0 <= x1 < x2 and y2 <= frame_h and x2 <= frame_w:
                    self.track_data_store[tid]['patch_prev'] = current_frame[y1:y2, x1:x2].copy()
                
                # Initialize history
                if tid not in self.track_history_store:
                    self.track_history_store[tid] = []
                
                # NEW: Initialize feature store
                if tid not in self.track_feature_store:
                    self.track_feature_store[tid] = []
            else:
                # Add current position to trail
                if 'trail' not in self.track_data_store[tid]:
                    self.track_data_store[tid]['trail'] = []
                
                self.track_data_store[tid]['trail'].append((cx, cy))
                
                # Keep trail to reasonable length
                max_trail_length = 20
                if len(self.track_data_store[tid]['trail']) > max_trail_length:
                    self.track_data_store[tid]['trail'].pop(0)
            
            # Update track data
            self.track_data_store[tid]['age'] += 1
            self.track_data_store[tid]['size_ratio'] = size_ratio
            self.track_data_store[tid]['is_front_facing'] = is_front_facing
            self.track_data_store[tid]['is_in_intersection'] = is_in_intersection
            
            # Skip if track is too new
            if self.track_data_store[tid]['age'] < self.min_track_age:
                results.append({
                    'track_id': tid,
                    'bbox': bbox,
                    'is_moving': False,
                    'confidence': 0.0,
                    'depth': 0.0,
                    'position_conf': 0.0,
                    'front_facing': is_front_facing,
                    'in_intersection': is_in_intersection
                })
                
                # Update track history
                self.track_history_store[tid].append({
                    'bbox': bbox,
                    'size_ratio': size_ratio,
                    'timestamp': timestamp,
                })
                
                # Keep history at reasonable length
                if len(self.track_history_store[tid]) > self.max_history:
                    self.track_history_store[tid].pop(0)
                    
                continue
            
            # 2. Apply parallax compensation with distance sensitivity - IMPROVED
            comp_dx, comp_dy, estimated_depth = self.apply_parallax_compensation(
                self.track_data_store[tid], camera_motion, frame_shape
            )
            
            # Store depth estimate
            self.track_data_store[tid]['depth'] = estimated_depth
            
            # Get previous bbox
            prev_bbox = self.track_data_store[tid]['bbox_prev']
            prev_x1, prev_y1, prev_x2, prev_y2 = prev_bbox
            prev_cx = (prev_x1 + prev_x2) / 2
            prev_cy = (prev_y1 + prev_y2) / 2
            
            # Calculate raw displacement
            raw_dx = cx - prev_cx
            raw_dy = cy - prev_cy
            
            # Apply camera motion compensation with parallax awareness
            rel_dx = raw_dx - comp_dx
            rel_dy = raw_dy - comp_dy
            
            # Calculate movement magnitude
            movement = np.sqrt(rel_dx**2 + rel_dy**2)
            
            # NEW: Apply non-linear boost to small movements to make them more detectable
            if movement < self.small_movement_boost_threshold:
                original_movement = movement
                boost_amount = (self.small_movement_boost_threshold - movement) * self.small_movement_boost_factor
                movement = movement + boost_amount
                
                if self.debug and abs(movement - original_movement) > 1.0:
                    print(f"  Small movement boost: {original_movement:.2f}px -> {movement:.2f}px (+{boost_amount:.2f})")
            
            # Calculate size change (can indicate front/rear motion)
            prev_width = prev_x2 - prev_x1
            prev_height = prev_y2 - prev_y1
            prev_area = prev_width * prev_height
            area_change = abs(obj_area - prev_area) / max(prev_area, 1)
            
            # NEW: Enhanced frontal movement detection
            if is_front_facing:
                # Lower threshold for frontal movement detection
                area_change_threshold = 0.01  # Just 1% change for frontal movement
                if area_change > area_change_threshold:
                    # Convert area change to equivalent movement for easier integration
                    diagonal = np.sqrt(obj_width**2 + obj_height**2)
                    area_movement = area_change * diagonal * 0.2
                    
                    # Use the larger of regular movement or area-based movement
                    original_movement = movement
                    movement = max(movement, area_movement)
                    
                    if self.debug and area_movement > original_movement:
                        print(f"  FRONTAL MOVEMENT: area_change={area_change:.4f}, using area_movement={area_movement:.2f}")
            
            # NEW: Compensate area change based on camera motion
            camera_approaching = False
            if camera_mag > self.camera_motion_threshold:  # If camera is moving significantly
                # Check if object is in center-ish of frame (camera likely moving toward/away)
                is_center = (cx > frame_w*0.3 and cx < frame_w*0.7)
                # If object is centered and camera moving, assume approaching relationship
                camera_approaching = is_center
                
                if camera_approaching:
                    # Store original area change for debug output
                    original_area_change = area_change
                    
                    # Subtract expected area change due to camera motion
                    expected_area_change = camera_mag * self.camera_motion_area_factor
                    area_change = max(0, area_change - expected_area_change)
                    
                    if self.debug and original_area_change > 0.02:
                        print(f"  Camera motion compensation: original={original_area_change:.4f}, adjusted={area_change:.4f}")
            
            # Debug output for tracked vehicles
            if self.debug and (estimated_depth > self.far_threshold or is_front_facing):
                print(f"Track {tid}: depth={estimated_depth:.2f}, movement={movement:.2f}px, area_change={area_change:.4f}")
                print(f"  Front facing: {is_front_facing}, In intersection: {is_in_intersection}, Camera approaching: {camera_approaching}")
                print(f"  Raw movement: dx={raw_dx:.2f}, dy={raw_dy:.2f}, Compensated: dx={comp_dx:.2f}, dy={comp_dy:.2f}")
            
            # ENHANCED: Much more sensitive position threshold for distant objects
            diagonal = np.sqrt(obj_width**2 + obj_height**2)
            
            # ENHANCED: Dramatically lower threshold for distant objects
            distance_factor = 1.0
            if estimated_depth > self.far_threshold:
                # Exponentially decrease threshold with distance - MORE AGGRESSIVE
                distance_factor = 1.0 - (self.distance_sensitivity * (estimated_depth - self.far_threshold))
                # Ensure it doesn't go negative - MORE AGGRESSIVE
                distance_factor = max(0.15, distance_factor)  # Changed from 0.2 to 0.15
            
            # Base position threshold calculation
            position_threshold = max(0.5, diagonal * 0.03 * distance_factor)
            
            # Special handling for vehicles at intersections
            if estimated_depth < 0.2 and is_in_intersection:
                # Much lower threshold for vehicles at intersections
                position_threshold *= self.intersection_threshold_factor
                if self.debug:
                    print(f"  INTERSECTION VEHICLE: lowered threshold to {position_threshold:.2f}")
            
            # Special handling for front-facing vehicles (approaching/receding)
            # Only if not explained by camera motion
            if is_front_facing and not camera_approaching:
                # Lower threshold for vehicles with headlights/taillights - MORE AGGRESSIVE
                position_threshold *= 0.6  # Changed from 0.7 to 0.6
                
                # If significant area change, boost the movement signal
                area_change_boost = min(1.0, area_change * 15.0)  # MORE AGGRESSIVE: Increased from 10.0 to 15.0
                movement += area_change_boost * diagonal * 0.05
                
                if self.debug and area_change_boost > 0.2:
                    print(f"  FRONT/REAR MOTION: area_change_boost={area_change_boost:.2f}, new movement={movement:.2f}")
            
            # Calculate position confidence
            position_confidence = min(1.0, movement / position_threshold)
            
            # Add boost for distant objects
            if estimated_depth > self.far_threshold:
                position_confidence = min(1.0, position_confidence + self.distant_object_boost)
            
            # Appearance-based confidence (SSIM)
            prev_patch = self.track_data_store[tid]['patch_prev']
            curr_patch = None
            if 0 <= y1 < y2 and 0 <= x1 < x2 and y2 <= frame_h and x2 <= frame_w:
                curr_patch = current_frame[y1:y2, x1:x2].copy()
            
            # Default appearance confidence (neutral)
            appearance_confidence = 0.5
            
            # Safe SSIM comparison
            if prev_patch is not None and curr_patch is not None:
                if prev_patch.shape[0] > 0 and prev_patch.shape[1] > 0 and curr_patch.shape[0] > 0 and curr_patch.shape[1] > 0:
                    try:
                        # Get minimum common dimensions
                        h = min(prev_patch.shape[0], curr_patch.shape[0])
                        w = min(prev_patch.shape[1], curr_patch.shape[1])
                        
                        if h >= 10 and w >= 10:  # Minimum size for meaningful comparison
                            # Resize patches to same size
                            p1 = cv2.resize(prev_patch, (w, h))
                            p2 = cv2.resize(curr_patch, (w, h))
                            
                            # Convert to grayscale
                            g1 = cv2.cvtColor(p1, cv2.COLOR_BGR2GRAY)
                            g2 = cv2.cvtColor(p2, cv2.COLOR_BGR2GRAY)
                            
                            # Calculate SSIM
                            from skimage.metrics import structural_similarity as ssim
                            ssim_score = ssim(g1, g2)
                            
                            # Adjust SSIM threshold based on characteristics
                            ssim_threshold = 0.7
                            if estimated_depth > self.far_threshold:
                                # Lower threshold for distant objects - MORE AGGRESSIVE
                                ssim_threshold = 0.7 - (estimated_depth * 0.5)  # Changed from 0.4 to 0.5
                            elif is_front_facing and not camera_approaching:
                                # Lower threshold for front-facing vehicles not due to camera - MORE AGGRESSIVE
                                ssim_threshold = 0.55  # Changed from 0.6 to 0.55
                                
                            # Transform to confidence (1.0 = different/moving, 0.0 = same/static)
                            appearance_confidence = max(0.0, min(1.0, (ssim_threshold - ssim_score) / ssim_threshold))
                            
                            # Boost confidence for distant or front-facing objects (not due to camera)
                            if estimated_depth > self.far_threshold:
                                appearance_confidence = min(1.0, appearance_confidence + self.distant_object_boost)
                            elif is_front_facing and not camera_approaching:
                                appearance_confidence = min(1.0, appearance_confidence + 0.2)
                    except Exception as e:
                        print(f"SSIM error: {e}")
            
            # 3. Track history and analyze trajectory
            # Update track history
            history_entry = {
                'bbox': bbox,
                'size_ratio': size_ratio,
                'timestamp': timestamp,
                'depth': estimated_depth,
                'movement': movement,
                'area_change': area_change,
                'is_front_facing': is_front_facing,
                'is_in_intersection': is_in_intersection,
                'camera_approaching': camera_approaching,
                'raw_dx': raw_dx,  # NEW: Store raw movement for debugging
                'raw_dy': raw_dy,
                'comp_dx': comp_dx,
                'comp_dy': comp_dy
            }
            
            self.track_history_store[tid].append(history_entry)
            if len(self.track_history_store[tid]) > self.max_history:
                self.track_history_store[tid].pop(0)
            
            # Analyze trajectory for consistent movement pattern
            trajectory_moving, trajectory_confidence = self.analyze_trajectory(
                tid, self.track_history_store[tid], self.camera_motion_history
            )
            
            # 4. Combine detection methods with adaptive weighting
            position_result = (position_confidence > 0.5, position_confidence)
            appearance_result = (appearance_confidence > 0.5, appearance_confidence)
            trajectory_result = (trajectory_moving, trajectory_confidence)
            
            # Special case for distant objects
            if estimated_depth > self.far_threshold:
                # If ANY detection method says it's moving, consider it moving - MORE AGGRESSIVE
                is_moving_position = position_confidence > 0.4  # Changed from 0.5 to 0.4
                is_moving_appearance = appearance_confidence > 0.4  # Changed from 0.5 to 0.4
                is_moving_trajectory = trajectory_moving
                
                if is_moving_position or is_moving_appearance or is_moving_trajectory:
                    # Short-circuit to true, with combined confidence
                    combined_moving = True
                    combined_confidence = (position_confidence + appearance_confidence + trajectory_confidence) / 3.0
                    combined_confidence = min(1.0, combined_confidence + self.distant_object_boost)
                    
                    if self.debug:
                        print(f"  DISTANT OBJECT movement detected! Combined conf: {combined_confidence:.2f}")
                else:
                    combined_moving, combined_confidence = self.combine_movement_detectors(
                        position_result, appearance_result, trajectory_result, 
                        self.track_data_store[tid]
                    )
            # Special case for front-facing vehicles (not due to camera) or intersection vehicles
            elif (is_front_facing and not camera_approaching) or (is_in_intersection and estimated_depth < 0.2):
                # Front-facing or intersection vehicles: be more lenient - MORE AGGRESSIVE
                is_moving_position = position_confidence > 0.35  # Changed from 0.4 to 0.35
                is_moving_appearance = appearance_confidence > 0.35  # Changed from 0.4 to 0.35
                is_moving_trajectory = trajectory_confidence > 0.35  # Changed from 0.4 to 0.35
                
                # More aggressive OR logic: if ANY strong indicator or TWO weak indicators
                if ((is_moving_position and position_confidence > 0.55) or  # Changed from 0.6 to 0.55
                   (is_moving_appearance and appearance_confidence > 0.55) or
                   (is_moving_trajectory and trajectory_confidence > 0.55) or
                   (sum([is_moving_position, is_moving_appearance, is_moving_trajectory]) >= 2)):
                    combined_moving = True
                    # Average confidence with boost
                    combined_confidence = (position_confidence + appearance_confidence + trajectory_confidence) / 3.0
                    boost = 0.25 if is_front_facing else 0.2  # Changed from 0.2/0.15 to 0.25/0.2
                    combined_confidence = min(1.0, combined_confidence + boost)
                    
                    if self.debug:
                        what_type = "FRONT-FACING" if is_front_facing else "INTERSECTION"
                        print(f"  {what_type} VEHICLE movement detected! Combined conf: {combined_confidence:.2f}")
                else:
                    combined_moving, combined_confidence = self.combine_movement_detectors(
                        position_result, appearance_result, trajectory_result, 
                        self.track_data_store[tid]
                    )
            else:
                # Regular combination for normal objects
                combined_moving, combined_confidence = self.combine_movement_detectors(
                    position_result, appearance_result, trajectory_result, 
                    self.track_data_store[tid]
                )
            
            # 5. Apply temporal consistency
            final_moving, final_confidence = self.apply_temporal_consistency(
                tid, combined_moving, combined_confidence, self.track_data_store
            )
            
            # Update consecutive state tracking
            if final_moving:
                self.track_data_store[tid]['consecutive_moving'] += 1
                self.track_data_store[tid]['consecutive_static'] = 0
            else:
                self.track_data_store[tid]['consecutive_static'] += 1
                self.track_data_store[tid]['consecutive_moving'] = 0
            
            # Special case persistence for distant objects
            if estimated_depth > self.far_threshold and not final_moving:
                # If it was previously moving recently, keep it moving - MORE AGGRESSIVE
                if self.track_data_store[tid]['consecutive_static'] < 4 and \
                   self.track_data_store[tid].get('was_moving_before', False):
                    final_moving = True
                    final_confidence = 0.65  # Changed from 0.6 to 0.65
                    if self.debug:
                        print(f"  Keeping distant object {tid} as moving due to history")
            
            # Special case persistence for front-facing vehicles (not due to camera)
            if is_front_facing and not camera_approaching and not final_moving:
                # Front-facing vehicles at intersections with recent movement - MORE AGGRESSIVE
                if self.track_data_store[tid]['consecutive_static'] < 3 and \
                   (self.track_data_store[tid].get('was_moving_before', False) or is_in_intersection):
                    final_moving = True
                    final_confidence = 0.65  # Changed from 0.6 to 0.65
                    if self.debug:
                        print(f"  Keeping front-facing vehicle {tid} as moving due to context")
            
            # Store if it was ever moving for future reference
            if final_moving:
                self.track_data_store[tid]['was_moving_before'] = True
            
            # Store results with all necessary fields directly in the dictionary
            results.append({
                'track_id': tid,
                'bbox': bbox,
                'is_moving': final_moving,
                'confidence': final_confidence,
                'depth': estimated_depth,
                'position_conf': position_confidence,
                'appearance_conf': appearance_confidence,
                'trajectory_conf': trajectory_confidence,
                'front_facing': is_front_facing,
                'in_intersection': is_in_intersection,
                'camera_approaching': camera_approaching,
                'raw_dx': raw_dx,  # NEW: Include raw movement data for visualization
                'raw_dy': raw_dy,
                'comp_dx': comp_dx,
                'comp_dy': comp_dy,
                'trail': self.track_data_store[tid]['trail']  # NEW: Include trail for visualization
            })
            
            # Update stored bbox and patch for next frame
            self.track_data_store[tid]['bbox_prev'] = bbox
            self.track_data_store[tid]['patch_prev'] = curr_patch
        
        return results, camera_motion

    def estimate_camera_motion_homography(self, prev_frame, curr_frame, detected_boxes):
        """Estimate camera motion using homography transformation."""
        # Convert to grayscale
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        
        # Create mask to exclude detected objects
        h, w = prev_gray.shape
        mask = np.ones((h, w), dtype=np.uint8) * 255
        
        # Mask out detected objects
        for box in detected_boxes:
            x1, y1, x2, y2 = map(int, box)
            # Add padding to ensure entire object is masked
            x1 = max(0, x1 - 10)
            y1 = max(0, y1 - 10)
            x2 = min(w, x2 + 10)
            y2 = min(h, y2 + 10)
            if 0 <= y1 < y2 <= h and 0 <= x1 < x2 <= w:
                mask[y1:y2, x1:x2] = 0
        
        # Find good features to track in background
        features = cv2.goodFeaturesToTrack(
            prev_gray, maxCorners=2000, qualityLevel=0.01, 
            minDistance=7, mask=mask, blockSize=7
        )
        
        if features is None or len(features) < 20:
            return None, 0, 0, None
        
        # Track features using Lucas-Kanade optical flow
        next_features, status, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray, curr_gray, features, None,
            winSize=(21, 21), maxLevel=3,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
        )
        
        # Keep only good matches
        good_old = features[status.flatten() == 1]
        good_new = next_features[status.flatten() == 1]
        
        if len(good_old) < 20:
            return None, 0, 0, None
        
        # Calculate flow vectors for visualization
        flow_vectors = good_new - good_old
        
        try:
            # Calculate homography matrix
            H, inliers = cv2.findHomography(
                good_old.reshape(-1, 2), 
                good_new.reshape(-1, 2), 
                cv2.RANSAC, 3.0
            )
            
            if H is None:
                return None, 0, 0, None
            
            # Extract translation component (approximation)
            dx = H[0, 2]
            dy = H[1, 2]
            
            # Calculate inlier ratio as quality measure
            inlier_ratio = np.sum(inliers) / len(inliers) if len(inliers) > 0 else 0
            
            # Check if camera motion is too large or inlier ratio is too low
            motion_magnitude = np.sqrt(dx**2 + dy**2)
            if motion_magnitude > 100 or inlier_ratio < 0.4:
                return None, 0, 0, None
            
            return H, dx, dy, {
                'inlier_ratio': inlier_ratio,
                'flow_vectors': flow_vectors,
                'magnitude': motion_magnitude
            }
        except Exception as e:
            print(f"Homography error: {e}")
            return None, 0, 0, None
    
    def apply_parallax_compensation(self, track_data, camera_motion, frame_shape):
        """Apply parallax-aware motion compensation based on object depth cues."""
        bbox = track_data["bbox_prev"]
        x1, y1, x2, y2 = bbox
        
        # Extract camera motion
        H = camera_motion.get('homography')
        dx = camera_motion.get('dx', 0)
        dy = camera_motion.get('dy', 0)
        
        # Calculate depth cues
        obj_width = x2 - x1
        obj_height = y2 - y1
        obj_area = obj_width * obj_height
        frame_area = frame_shape[0] * frame_shape[1]
        area_ratio = obj_area / frame_area
        
        # Position-based depth cue (objects at bottom of frame are generally closer)
        bottom_position = y2 / frame_shape[0]  # 0 at top, 1 at bottom
        
        # Check for front-facing attribute
        is_front_facing = track_data.get('is_front_facing', False)
        
        # Enhanced depth estimation
        # Size has higher weight for small objects
        size_weight = 0.7
        position_weight = 0.3
        
        # Size-based depth (small objects are far)
        size_depth = 1 - min(1.0, area_ratio * 40)  # Higher multiplier to be more sensitive
        
        # Position-based depth (lower in frame = closer)
        position_depth = 1 - bottom_position
        
        # Combined depth
        estimated_depth = size_weight * size_depth + position_weight * position_depth
        
        # Adjust depth for front-facing vehicles (they appear larger)
        if is_front_facing and bottom_position > 0.6:  # Lower in frame
            # Front-facing cars in lower part of frame are closer
            estimated_depth = max(0.1, estimated_depth * 0.8)
        
        # Clamp depth to reasonable range
        estimated_depth = max(0.1, min(0.95, estimated_depth))
        
        # More limited parallax effect to avoid overcompensation - IMPROVED
        # Calculate parallax factor (how much camera motion affects this object)
        parallax_factor = 1.0 - (0.2 * estimated_depth)  # CHANGED: Reduced from 0.3 to 0.2
        
        # Adjust compensation based on depth
        if H is not None:
            # For homography, we'll transform the center point
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            
            try:
                # Create point array
                pts = np.array([[[cx, cy]]], dtype=np.float32)
                
                # Apply homography transform
                transformed_pts = cv2.perspectiveTransform(pts, H)
                expected_cx = transformed_pts[0][0][0]
                expected_cy = transformed_pts[0][0][1]
                
                # Calculate expected motion
                exp_dx = expected_cx - cx
                exp_dy = expected_cy - cy
                
                # Apply parallax factor
                comp_dx = exp_dx * parallax_factor
                comp_dy = exp_dy * parallax_factor
            except Exception as e:
                print(f"Parallax compensation error: {e}")
                # Fall back to simple translation
                comp_dx = dx * parallax_factor
                comp_dy = dy * parallax_factor
        else:
            # For simple translation, adjust by parallax factor
            comp_dx = dx * parallax_factor
            comp_dy = dy * parallax_factor
        
        return comp_dx, comp_dy, estimated_depth
    
    def analyze_trajectory(self, track_id, track_history, camera_motions, min_frames=5):
        """Analyze object trajectory over multiple frames to identify real movement."""
        if len(track_history) < min_frames:
            return False, 0.0
        
        # Extract position history (most recent min_frames frames)
        positions = []
        timestamps = []
        depths = []
        front_facing = []
        area_changes = []
        camera_approaching = []
        raw_movements = []  # NEW: Track raw movements before compensation
        
        for i in range(min(min_frames, len(track_history))):
            frame_data = track_history[-(i+1)]
            bbox = frame_data['bbox']
            cx = (bbox[0] + bbox[2]) / 2
            cy = (bbox[1] + bbox[3]) / 2
            positions.append((cx, cy))
            timestamps.append(frame_data.get('timestamp', i))
            depths.append(frame_data.get('depth', 0.5))
            front_facing.append(frame_data.get('is_front_facing', False))
            area_changes.append(frame_data.get('area_change', 0))
            camera_approaching.append(frame_data.get('camera_approaching', False))
            
            # NEW: Store raw movements
            if 'raw_dx' in frame_data and 'raw_dy' in frame_data:
                raw_movements.append((frame_data['raw_dx'], frame_data['raw_dy']))
        
        # Extract movement history
        movement_history = []
        for i in range(min(min_frames, len(track_history))):
            if 'movement' in track_history[-(i+1)]:
                movement_history.append(track_history[-(i+1)]['movement'])
        
        # Calculate avg depth and other attributes
        avg_depth = np.mean(depths)
        is_front_facing = any(front_facing)
        avg_area_change = np.mean(area_changes) if area_changes else 0
        avg_camera_approaching = np.mean([1 if ca else 0 for ca in camera_approaching]) > 0.5
        
        # Transform all positions to current frame coordinates
        # This compensates for camera motion
        camera_compensated_positions = []
        
        # Start with current position (no transformation needed)
        camera_compensated_positions.append(positions[0])
        
        # Transform previous positions using stored camera motions
        for i in range(1, len(positions)):
            prev_pos = positions[i]
            
            # Apply all camera motions from frame i to current frame
            transformed_x, transformed_y = prev_pos[0], prev_pos[1]
            
            for j in range(i):
                motion_idx = len(camera_motions) - i + j
                if motion_idx >= 0 and motion_idx < len(camera_motions):
                    motion = camera_motions[motion_idx]
                    H = motion.get('homography')
                    
                    if H is not None:
                        try:
                            # Apply homography transformation
                            pts = np.array([[[transformed_x, transformed_y]]], dtype=np.float32)
                            transformed_pts = cv2.perspectiveTransform(pts, H)
                            transformed_x = transformed_pts[0][0][0]
                            transformed_y = transformed_pts[0][0][1]
                        except Exception as e:
                            print(f"Trajectory transformation error: {e}")
                            # Fallback to simple translation
                            transformed_x += motion.get('dx', 0)
                            transformed_y += motion.get('dy', 0)
                    else:
                        # Apply simple translation
                        transformed_x += motion.get('dx', 0)
                        transformed_y += motion.get('dy', 0)
            
            camera_compensated_positions.append((transformed_x, transformed_y))
        
        # Convert to numpy array for analysis
        positions_array = np.array(camera_compensated_positions)
        timestamps_array = np.array(timestamps)
        
        # If all positions are very close to each other, object is stationary
        if len(positions_array) < 3:
            return False, 0.0
        
        # Calculate total displacement
        total_displacement = 0
        for i in range(1, len(positions_array)):
            dx = positions_array[i-1][0] - positions_array[i][0]
            dy = positions_array[i-1][1] - positions_array[i][1]
            total_displacement += np.sqrt(dx*dx + dy*dy)
        
        # NEW: Calculate total raw displacement (without camera compensation)
        total_raw_displacement = 0
        if len(raw_movements) >= 2:
            for i in range(len(raw_movements)):
                raw_dx, raw_dy = raw_movements[i]
                total_raw_displacement += np.sqrt(raw_dx**2 + raw_dy**2)
            
            # Average raw displacement per frame
            avg_raw_displacement = total_raw_displacement / len(raw_movements)
            
            # Debug output for distant objects
            if self.debug and avg_depth > self.far_threshold:
                print(f"  Raw displacement avg: {avg_raw_displacement:.2f}px per frame")
        
        # Debug output for interesting objects
        if self.debug and (avg_depth > self.far_threshold or is_front_facing):
            print(f"  Trajectory analysis: total_displacement={total_displacement:.2f}px, avg_area_change={avg_area_change:.4f}")
            if movement_history:
                print(f"  Recent movements: {', '.join([f'{m:.2f}' for m in movement_history])}")
                print(f"  Camera approaching: {avg_camera_approaching}")
        
        # Special case for front-facing vehicles with area changes not due to camera
        if is_front_facing and avg_area_change > 0.01 and not avg_camera_approaching:  # CHANGED: Threshold from 0.02 to 0.01
            # Front/rear motion detected via size change
            confidence = min(1.0, avg_area_change * 25.0)  # CHANGED: Scale from 20.0 to 25.0
            if self.debug:
                print(f"  FRONT/REAR MOTION detected via size change: confidence={confidence:.2f}")
            return True, min(1.0, confidence + 0.25)  # CHANGED: Boost from 0.2 to 0.25
        
        # Fast path for distant objects with clear motion
        if avg_depth > self.far_threshold and total_displacement > 4.0:  # CHANGED: Threshold from 5.0 to 4.0
            confidence = min(1.0, total_displacement / 8.0)  # CHANGED: Scale from 10.0 to 8.0
            return True, confidence + self.distant_object_boost  # Add boost
        
        # Calculate trajectory linearity (R²) by fitting line to points
        try:
            # Time differences
            time_diffs = timestamps_array - timestamps_array[-1]
            
            # Fit linear model for x and y coordinates
            x_coords = positions_array[:, 0]
            y_coords = positions_array[:, 1]
            
            # Polynomial fit
            px = np.polyfit(time_diffs, x_coords, 1)
            py = np.polyfit(time_diffs, y_coords, 1)
            
            # Evaluate fit
            x_pred = np.polyval(px, time_diffs)
            y_pred = np.polyval(py, time_diffs)
            
            # Calculate R² to measure linearity
            x_mean = np.mean(x_coords)
            y_mean = np.mean(y_coords)
            
            # Total sum of squares
            ss_tot_x = np.sum((x_coords - x_mean)**2)
            ss_tot_y = np.sum((y_coords - y_mean)**2)
            
            # Residual sum of squares
            ss_res_x = np.sum((x_coords - x_pred)**2)
            ss_res_y = np.sum((y_coords - y_pred)**2)
            
            # R² calculation
            if ss_tot_x > 0 and ss_tot_y > 0:
                r2_x = 1 - (ss_res_x / ss_tot_x)
                r2_y = 1 - (ss_res_y / ss_tot_y)
                r2 = (r2_x + r2_y) / 2
            else:
                r2 = 0
            
            # Calculate velocity from linear fit (slope of line)
            velocity_x = px[0]
            velocity_y = py[0]
            velocity = np.sqrt(velocity_x**2 + velocity_y**2)
            
            # DEBUG: Print velocity and R² for interesting objects
            if self.debug and (avg_depth > self.far_threshold or is_front_facing):
                print(f"  Trajectory velocity={velocity:.4f}, R²={r2:.4f}")
            
            # Size factor is less impactful for distant objects
            size_factor = 5.0
            if avg_depth > self.far_threshold:
                # For distant objects, use a very small fixed size factor
                size_factor = 1.0
            elif is_front_facing and not avg_camera_approaching:
                # For front-facing objects not due to camera, use a smaller fixed size factor
                size_factor = 2.0
            else:
                # For closer objects, calculate based on size
                size_factor = track_history[-1].get('size_ratio', 0.01) * 5.0
            
            # Threshold calculation dramatically favors distant objects
            depth_factor = 1.0
            if avg_depth > self.far_threshold:
                depth_factor = 1.0 - (avg_depth - self.far_threshold) * self.distance_sensitivity
                depth_factor = max(0.1, depth_factor)  # Ensure it doesn't go too low
            elif is_front_facing and not avg_camera_approaching:
                # Lower threshold for front-facing vehicles not due to camera
                depth_factor = 0.6  # CHANGED: Threshold from 0.7 to 0.6
            
            # Very small threshold for distant objects
            threshold = 0.5 + size_factor * depth_factor
            
            # Heavy weighting for trajectory consistency for distant objects
            linearity_weight = 0.5  # Base weight for path linearity
            velocity_weight = 0.5   # Base weight for absolute velocity
            
            if avg_depth > self.far_threshold:
                # For distant objects, favor linearity more (consistent direction)
                linearity_weight = 0.7
                velocity_weight = 0.3
            elif is_front_facing and not avg_camera_approaching:
                # For front-facing vehicles not due to camera, favor velocity more
                linearity_weight = 0.3
                velocity_weight = 0.7
            
            # Depth boost for special cases
            depth_boost = 0.0
            if avg_depth > self.far_threshold:
                depth_boost = self.distant_object_boost
                # Additional boost if R² is good (consistent direction)
                if r2 > 0.5:
                    depth_boost += 0.15  # CHANGED: Boost from 0.1 to 0.15
            elif is_front_facing and not avg_camera_approaching:
                depth_boost = 0.2  # CHANGED: Boost from 0.15 to 0.2
                # Additional boost for significant area change
                if avg_area_change > 0.01:  # CHANGED: Threshold from 0.01 to 0.008
                    depth_boost += min(0.25, avg_area_change * 12.0)  # CHANGED: Boost from 0.2 to 0.25, factor from 10.0 to 12.0
            
            # Combine all factors
            confidence = (
                linearity_weight * min(1.0, max(0, r2)) + 
                velocity_weight * min(1.0, velocity / threshold) +
                depth_boost
            )
            
            # Ensure confidence is in valid range
            confidence = min(1.0, max(0.0, confidence))
            
            # Much lower threshold for distant objects or front-facing vehicles not due to camera
            motion_threshold = 0.55  # CHANGED: Lowered from 0.6 to 0.55
            if avg_depth > self.far_threshold:
                motion_threshold = 0.35  # CHANGED: Lowered from 0.4 to 0.35
            elif is_front_facing and not avg_camera_approaching:
                motion_threshold = 0.4  # CHANGED: Lowered from 0.45 to 0.4
            
            is_moving = confidence > motion_threshold
            
            return is_moving, confidence
            
        except Exception as e:
            print(f"Trajectory analysis error: {e}")
            # If fitting fails, fall back to simple displacement check
            avg_displacement = total_displacement / (len(positions_array) - 1)
            
            # Much more lenient threshold for distant objects or front-facing vehicles not due to camera
            threshold = 1.8  # CHANGED: From 2.0 to 1.8
            if avg_depth > self.far_threshold:
                threshold = 0.4  # CHANGED: From 0.5 to 0.4
            elif is_front_facing and not avg_camera_approaching:
                threshold = 0.8  # CHANGED: From 1.0 to 0.8
                
            confidence = min(1.0, avg_displacement / threshold)
            
            # Add boost for special cases
            if avg_depth > self.far_threshold:
                confidence += self.distant_object_boost
                confidence = min(1.0, confidence)
            elif is_front_facing and not avg_camera_approaching:
                confidence += 0.2  # CHANGED: Boost from 0.15 to 0.2
                confidence = min(1.0, confidence)
            
            return confidence > 0.5, confidence
    
    def combine_movement_detectors(self, position_result, appearance_result, trajectory_result, track_data):
        """Combine multiple movement detection methods with adaptive weighting."""
        # Unpack results
        pos_moving, pos_conf = position_result
        app_moving, app_conf = appearance_result
        traj_moving, traj_conf = trajectory_result
        
        # Get track characteristics
        track_age = track_data.get('age', 0)
        size_ratio = track_data.get('size_ratio', 0.01)
        estimated_depth = track_data.get('depth', 0.5)
        is_front_facing = track_data.get('is_front_facing', False)
        is_in_intersection = track_data.get('is_in_intersection', False)
        # Check camera approaching
        camera_approaching = track_data.get('camera_approaching', False)
        
        # NEW: Special handling for new tracks - be more aggressive
        if track_age < 10:  # New track
            # Boost any movement confidence that shows signs of movement
            if pos_conf > 0.4:
                pos_conf = min(1.0, pos_conf + 0.15)
            if app_conf > 0.4:
                app_conf = min(1.0, app_conf + 0.15)
            if traj_conf > 0.4:
                traj_conf = min(1.0, traj_conf + 0.15)
        
        # Completely revised weighting system for different cases
        if estimated_depth > self.far_threshold:  # Far object
            # For distant objects, hugely favor ANY sign of movement
            if pos_moving and pos_conf > 0.55:  # CHANGED: From 0.6 to 0.55
                pos_weight = 0.8
                app_weight = 0.1
                traj_weight = 0.1
            elif traj_moving and traj_conf > 0.55:  # CHANGED: From 0.6 to 0.55
                pos_weight = 0.2
                app_weight = 0.1
                traj_weight = 0.7
            elif app_moving and app_conf > 0.55:  # CHANGED: From 0.6 to 0.55
                pos_weight = 0.1
                app_weight = 0.8
                traj_weight = 0.1
            else:
                # No strong indicator, use more balanced weights but favor trajectory
                pos_weight = 0.3
                app_weight = 0.2
                traj_weight = 0.5
        elif is_front_facing and not camera_approaching:  # Front-facing vehicle (not due to camera)
            # For front-facing vehicles, favor trajectory more
            pos_weight = 0.3
            app_weight = 0.2
            traj_weight = 0.5
        elif estimated_depth < 0.2:  # Very close object
            # For very close objects, especially at intersections
            if is_in_intersection:
                # At intersections, trajectory is most important
                pos_weight = 0.3
                app_weight = 0.2
                traj_weight = 0.5
            else:
                # For close objects, trajectory is more reliable than position
                pos_weight = 0.4
                app_weight = 0.1
                traj_weight = 0.5
        else:
            # Normal logic for medium distance objects
            if track_age < 5:
                # For new tracks, rely more on position and appearance
                pos_weight = 0.5
                app_weight = 0.4
                traj_weight = 0.1
            else:
                # For established tracks, trajectory becomes more important
                track_age_factor = min(1.0, (track_age - 5) / 15)
                traj_weight = 0.3 + (0.4 * track_age_factor)
                
                if size_ratio > 0.05:  # Very close/large objects
                    # Position is more reliable than appearance due to self-occlusion
                    pos_weight = 0.5
                    app_weight = 0.1
                else:
                    # For smaller, medium-distance objects, balance position and appearance
                    pos_weight = 0.4
                    app_weight = 0.2
        
        # Ensure weights sum to 1.0
        total_weight = pos_weight + app_weight + traj_weight
        pos_weight /= total_weight
        app_weight /= total_weight
        traj_weight /= total_weight
        
        # Apply confidence boosting
        if estimated_depth > self.far_threshold:
            # For distant objects, dramatically boost all movement confidences
            boost = self.distant_object_boost * 2.5  # CHANGED: From 2.0 to 2.5
            
            # Only boost if at least one detector says it's moving
            if pos_moving or app_moving or traj_moving:
                pos_conf = min(1.0, pos_conf + boost) if pos_moving else pos_conf
                app_conf = min(1.0, app_conf + boost) if app_moving else app_conf
                traj_conf = min(1.0, traj_conf + boost) if traj_moving else traj_conf
        elif is_front_facing and not camera_approaching:
            # For front-facing vehicles not due to camera, boost movement confidences
            boost = 0.25  # CHANGED: From 0.2 to 0.25
            
            # Only boost if at least one detector says it's moving
            if pos_moving or app_moving or traj_moving:
                pos_conf = min(1.0, pos_conf + boost) if pos_moving else pos_conf
                app_conf = min(1.0, app_conf + boost) if app_moving else app_conf
                traj_conf = min(1.0, traj_conf + boost) if traj_moving else traj_conf
        
        # Combine weighted confidences
        combined_conf = (
            pos_weight * pos_conf +
            app_weight * app_conf +
            traj_weight * traj_conf
        )
        
        # Dramatically lowered threshold for special cases
        if track_data.get('is_moving', False):
            # Currently moving - need less evidence to stay moving
            threshold = 0.4  # No change
        else:
            # Currently static - need more evidence to become moving
            if estimated_depth > self.far_threshold:
                threshold = 0.35  # CHANGED: From 0.4 to 0.35
            elif is_front_facing and not camera_approaching:
                threshold = 0.4  # CHANGED: From 0.45 to 0.4
            elif is_in_intersection and estimated_depth < 0.2:
                threshold = 0.45  # CHANGED: From 0.5 to 0.45
            else:
                threshold = 0.55   # CHANGED: From 0.6 to 0.55
        
        # Debug for special objects
        if self.debug and (estimated_depth > self.far_threshold or is_front_facing):
            print(f"  Combined confidence: {combined_conf:.2f}, threshold: {threshold:.2f}")
            print(f"  Weights: pos={pos_weight:.2f}, app={app_weight:.2f}, traj={traj_weight:.2f}")
            print(f"  Confs: pos={pos_conf:.2f}, app={app_conf:.2f}, traj={traj_conf:.2f}")
            if camera_approaching:
                print(f"  Camera approaching - using more conservative detection")
        
        is_moving = combined_conf > threshold
        
        return is_moving, combined_conf
    
    def apply_temporal_consistency(self, track_id, is_moving, confidence, track_data_store):
        """Apply temporal filtering to reduce classification flickering."""
        # Initialize track data if it doesn't exist
        if 'state_history' not in track_data_store[track_id]:
            track_data_store[track_id]['state_history'] = []
            track_data_store[track_id]['confidence_history'] = []
            track_data_store[track_id]['stability_counter'] = 0
        
        track_data = track_data_store[track_id]
        
        # Update state and confidence history
        track_data['state_history'].append(int(is_moving))
        track_data['confidence_history'].append(confidence)
        
        # Keep history at reasonable length (last 15 frames)
        max_history = 15
        if len(track_data['state_history']) > max_history:
            track_data['state_history'] = track_data['state_history'][-max_history:]
            track_data['confidence_history'] = track_data['confidence_history'][-max_history:]
        
        # Require minimum history
        if len(track_data['state_history']) < 3:
            track_data['is_moving'] = is_moving
            return is_moving, confidence
        
        # Calculate weighted average (recent frames and high confidence get more weight)
        total_weight = 0
        weighted_sum = 0
        
        # Create recency weights (newer frames have higher weight)
        history_len = len(track_data['state_history'])
        recency_weights = np.linspace(0.5, 1.0, history_len)
        
        for i in range(history_len):
            # Combined weight from recency and confidence
            weight = recency_weights[i] * track_data['confidence_history'][i]
            weighted_sum += track_data['state_history'][i] * weight
            total_weight += weight
        
        # Calculate weighted average
        if total_weight > 0:
            weighted_avg = weighted_sum / total_weight
        else:
            weighted_avg = 0.5
        
        # Get current state and characteristics
        current_state = track_data['is_moving']
        estimated_depth = track_data.get('depth', 0.5)
        is_distant = estimated_depth > self.far_threshold
        is_front_facing = track_data.get('is_front_facing', False)
        is_in_intersection = track_data.get('is_in_intersection', False)
        # Check camera approaching
        camera_approaching = track_data.get('camera_approaching', False)
        
        # Customized thresholds based on vehicle characteristics
        if is_distant:
            # Distant objects need almost no temporal consistency
            moving_to_static_threshold = 0.15   # CHANGED: From 0.2 to 0.15
            static_to_moving_threshold = 0.35   # CHANGED: From 0.4 to 0.35
            required_evidence_count = 1        # Only need one frame to switch
        elif is_front_facing and not camera_approaching:
            # Front-facing vehicles need less temporal consistency
            moving_to_static_threshold = 0.2  # CHANGED: From 0.25 to 0.2
            static_to_moving_threshold = 0.45   # CHANGED: From 0.5 to 0.45
            required_evidence_count = 2        # Need two frames to switch
        elif is_in_intersection and estimated_depth < 0.2:
            # Vehicles at intersections
            moving_to_static_threshold = 0.25  # CHANGED: From 0.3 to 0.25
            static_to_moving_threshold = 0.55  # CHANGED: From 0.6 to 0.55
            required_evidence_count = 2
        else:
            # Standard thresholds for normal objects
            moving_to_static_threshold = 0.25  # CHANGED: From 0.3 to 0.25
            static_to_moving_threshold = 0.65  # CHANGED: From 0.7 to 0.65
            required_evidence_count = 3
        
        # Debug for interesting objects
        if self.debug and (is_distant or is_front_facing):
            print(f"  Temporal weighted_avg: {weighted_avg:.2f}")
            print(f"  Current state: {'moving' if current_state else 'static'}")
            print(f"  Required evidence: {required_evidence_count}")
            if camera_approaching:
                print(f"  Camera approaching - using standard temporal consistency")
        
        # Strong evidence for state change
        if current_state and weighted_avg < moving_to_static_threshold:  # Moving -> Static
            if track_data['stability_counter'] < -required_evidence_count:  # Require consistent evidence
                new_state = False
                track_data['stability_counter'] = 0
            else:
                new_state = current_state
                track_data['stability_counter'] -= 1
        elif not current_state and weighted_avg > static_to_moving_threshold:  # Static -> Moving
            if track_data['stability_counter'] > required_evidence_count:  # Require consistent evidence
                new_state = True
                track_data['stability_counter'] = 0
            else:
                new_state = current_state
                track_data['stability_counter'] += 1
                
                # Fast path for special objects to become moving
                fast_transition = False
                if is_distant and track_data['stability_counter'] >= required_evidence_count:
                    fast_transition = True
                    what_type = "DISTANT"
                elif is_front_facing and not camera_approaching and track_data['stability_counter'] >= required_evidence_count:
                    fast_transition = True
                    what_type = "FRONT-FACING"
                
                if fast_transition:
                    new_state = True
                    track_data['stability_counter'] = 0
                    if self.debug:
                        print(f"  {what_type} OBJECT switched to moving!")
                    
        # Moderate evidence for state change
        elif current_state and weighted_avg < 0.4:  # Some evidence for Moving -> Static
            # Make special objects VERY reluctant to switch to static
            if is_distant:
                track_data['stability_counter'] -= 0.4  # CHANGED: From 0.5 to 0.4
            elif is_front_facing and not camera_approaching:
                track_data['stability_counter'] -= 0.6  # CHANGED: From 0.7 to 0.6
            else:
                track_data['stability_counter'] -= 1
                
            # Higher threshold for special objects to switch to static
            static_threshold = -5
            if is_distant:
                static_threshold = -9  # CHANGED: From -8 to -9
            elif is_front_facing and not camera_approaching:
                static_threshold = -7  # CHANGED: From -6 to -7
                
            new_state = track_data['stability_counter'] < static_threshold
        elif not current_state and weighted_avg > 0.6:  # Some evidence for Static -> Moving
            # Make special objects VERY eager to switch to moving
            if is_distant:
                track_data['stability_counter'] += 2.5  # CHANGED: From 2.0 to 2.5
            elif is_front_facing and not camera_approaching:
                track_data['stability_counter'] += 2.0  # CHANGED: From 1.5 to 2.0
            else:
                track_data['stability_counter'] += 1
                
            # Lower threshold for special objects to switch to moving
            moving_threshold = 3
            if is_distant:
                moving_threshold = 1  # No change
            elif is_front_facing and not camera_approaching:
                moving_threshold = 1  # CHANGED: From 2 to 1
                
            new_state = track_data['stability_counter'] > moving_threshold
        # Not enough evidence for state change
        else:
            track_data['stability_counter'] = 0  # Reset counter
            new_state = current_state
        
        # Limit counter range
        track_data['stability_counter'] = max(-10, min(10, track_data['stability_counter']))
        
        # Update state
        track_data['is_moving'] = new_state
        
        # Calculate new confidence (higher for stable classifications)
        stable_confidence = confidence * (1.0 + min(1.0, abs(track_data['stability_counter']) / 5) * 0.2)
        stable_confidence = min(1.0, stable_confidence)
        
        # Add a boost for special moving objects to maintain high confidence
        if is_distant and new_state:
            stable_confidence = min(1.0, stable_confidence + 0.15)  # CHANGED: From 0.1 to 0.15
        elif is_front_facing and not camera_approaching and new_state:
            stable_confidence = min(1.0, stable_confidence + 0.15)  # CHANGED: From 0.1 to 0.15
        
        return new_state, stable_confidence


def crop_patch(frame, bbox):
    """Extract image patch from frame using bounding box coordinates."""
    x1, y1, x2, y2 = map(int, bbox)
    
    # Ensure coordinates are within frame boundaries
    h, w = frame.shape[:2]
    x1 = max(0, min(x1, w-1))
    y1 = max(0, min(y1, h-1))
    x2 = max(0, min(x2, w-1))
    y2 = max(0, min(y2, h-1))
    
    # Check if resulting patch is valid
    if x2 <= x1 or y2 <= y1:
        return None
    
    return frame[y1:y2, x1:x2]


def draw_bounding_box(frame, bbox, label, is_moving, confidence, trail=None):
    """Draw an improved bounding box with movement information and trail."""
    x1, y1, x2, y2 = map(int, bbox)
    
    # Ensure coordinates are within frame boundaries
    h, w = frame.shape[:2]
    x1 = max(0, min(x1, w-1))
    y1 = max(0, min(y1, h-1))
    x2 = max(0, min(x2, w-1))
    y2 = max(0, min(y2, h-1))
    
    # Skip invalid boxes
    if x2 <= x1 or y2 <= y1:
        return
    
    # Color based on movement state with intensity based on confidence
    if is_moving:
        # Red with intensity based on confidence
        intensity = min(255, int(200 + 55 * confidence))
        color = (0, 0, intensity)  
    else:
        # Green with intensity based on confidence
        intensity = min(255, int(200 + 55 * (1 - confidence)))
        color = (0, intensity, 0)
    
    # Draw trail if available
    if trail is not None and len(trail) > 1:
        for i in range(1, len(trail)):
            pt1 = (int(trail[i-1][0]), int(trail[i-1][1]))
            pt2 = (int(trail[i][0]), int(trail[i][1]))
            
            # Skip invalid points
            if (pt1[0] < 0 or pt1[0] >= w or pt1[1] < 0 or pt1[1] >= h or
                pt2[0] < 0 or pt2[0] >= w or pt2[1] < 0 or pt2[1] >= h):
                continue
            
            # Draw trail line with fading intensity based on age
            alpha = i / len(trail)  # 0 for oldest, 1 for newest
            trail_color = tuple([int(c * alpha) for c in color])
            cv2.line(frame, pt1, pt2, trail_color, 2, cv2.LINE_AA)
    
    # Draw solid outline
    thickness = 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    
    # Add confidence to label if available
    if confidence is not None:
        label = f"{label} ({confidence:.2f})"
    
    # Create background for text
    text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    text_w, text_h = text_size
    
    # Draw text background
    cv2.rectangle(frame, (x1, y1 - text_h - 8), (x1 + text_w + 10, y1), color, -1)
    
    # Draw text with white color on the colored background
    cv2.putText(frame, label, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 
                0.6, (255, 255, 255), 2, cv2.LINE_AA)


def draw_movement_vector(frame, bbox, raw_dx, raw_dy, comp_dx, comp_dy):
    """Draw vectors showing raw movement and compensated movement."""
    x1, y1, x2, y2 = map(int, bbox)
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2
    
    # Ensure center is within frame
    h, w = frame.shape[:2]
    if center_x < 0 or center_x >= w or center_y < 0 or center_y >= h:
        return
    
    # Scale factor to make vectors visible
    scale = 3.0
    
    # Draw raw movement vector (white)
    end_x = int(center_x + raw_dx * scale)
    end_y = int(center_y + raw_dy * scale)
    
    # Ensure end point is within frame
    end_x = max(0, min(end_x, w-1))
    end_y = max(0, min(end_y, h-1))
    
    cv2.arrowedLine(frame, (center_x, center_y), (end_x, end_y), 
                    (255, 255, 255), 2, cv2.LINE_AA)
    
    # Draw compensated movement vector (yellow)
    end_x = int(center_x + (raw_dx - comp_dx) * scale)
    end_y = int(center_y + (raw_dy - comp_dy) * scale)
    
    # Ensure end point is within frame
    end_x = max(0, min(end_x, w-1))
    end_y = max(0, min(end_y, h-1))
    
    cv2.arrowedLine(frame, (center_x, center_y), (end_x, end_y), 
                    (0, 255, 255), 2, cv2.LINE_AA)


def draw_motion_overlay(frame, camera_motion):
    """Draw camera motion information overlay."""
    h, w = frame.shape[:2]
    
    if camera_motion is None or 'dx' not in camera_motion or 'dy' not in camera_motion:
        return
    
    dx = camera_motion.get('dx', 0)
    dy = camera_motion.get('dy', 0)
    motion_data = camera_motion.get('data', {})
    
    # Draw arrow indicating camera motion direction and magnitude
    center_x, center_y = 100, 100  # Position in top-left corner
    # Scale factor to make arrow visible but not too large
    scale = 2.0
    end_x = int(center_x + dx * scale)
    end_y = int(center_y + dy * scale)
    
    # Draw background circle
    cv2.circle(frame, (center_x, center_y), 50, (0, 0, 0), -1)
    
    # Draw arrow
    cv2.arrowedLine(frame, (center_x, center_y), (end_x, end_y), 
                   (0, 165, 255), 3, cv2.LINE_AA, tipLength=0.3)
    
    # Draw magnitude text
    if motion_data is not None:
        magnitude = motion_data.get('magnitude', 0)
        inlier_ratio = motion_data.get('inlier_ratio', 0)
        
        cv2.putText(frame, f"Camera Motion: {magnitude:.1f}px", 
                   (150, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"Conf: {inlier_ratio:.2f}", 
                   (150, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2, cv2.LINE_AA)


def main():
    # Load YOLOv8 with higher confidence threshold for better detection
    model = YOLO("yolov8n.pt")
    
    # Set detection parameters
    conf_threshold = 0.35  # Higher confidence threshold for more reliable detections
    
    # Initialize DeepSORT with IMPROVED parameters for vehicle tracking
    tracker = DeepSort(
        max_age=40,        # CHANGED: Increased from 30 to 40
        n_init=2,          # CHANGED: Reduced from 3 to 2
        max_iou_distance=0.8,  # CHANGED: Increased from 0.7 to 0.8
        max_cosine_distance=0.7  # CHANGED: Increased from 0.5 to 0.7
    )

    # Initialize our advanced movement detector
    movement_detector = AdvancedMovementDetector()
    
    # Print initialization message
    print("Starting Enhanced Movement Detector with Camera Motion Compensation")
    print(f"Distance sensitivity: {movement_detector.distance_sensitivity}")
    print(f"Distant object boost: {movement_detector.distant_object_boost}")
    print(f"Far threshold: {movement_detector.far_threshold}")
    print(f"Headlight threshold: {movement_detector.headlight_brightness_threshold}")
    print(f"Intersection threshold factor: {movement_detector.intersection_threshold_factor}")
    print(f"Camera motion area factor: {movement_detector.camera_motion_area_factor}")
    print(f"Small movement boost threshold: {movement_detector.small_movement_boost_threshold}")
    print(f"Small movement boost factor: {movement_detector.small_movement_boost_factor}")

    # Video input/output
    cap = cv2.VideoCapture("man1.mp4")
    if not cap.isOpened():
        print("Error: Could not open video file")
        return
        
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter("output_enhanced.mp4", fourcc, fps, (width, height))

    # Read first frame
    ret, prev_frame = cap.read()
    if not ret:
        print("Error reading video.")
        return

    frame_count = 0
    
    # Stats for dashboard
    stats = {
        'total': 0,
        'moving': 0,
        'static': 0,
        'frame': 0,
        'fps': fps,
        'camera_motion': 0,
        'front_facing': 0,
        'camera_approaching': 0
    }

    # Define the target class to detect (class 5 for vehicles)
    target_class = 5
    
    # NEW: Track ID management
    last_track_features = {}
    stored_track_features = {}

    while True:
        ret, curr_frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        print(f"Processing frame {frame_count}", end="\r")

        # Detect objects in current frame
        results = model(curr_frame, conf=conf_threshold)[0]
        
        # Extract detection information
        boxes = []
        vehicle_boxes = []
        
        # Filter for vehicle class (5)
        for i, cls in enumerate(results.boxes.cls):
            if int(cls) == 5:  # explicitly check for class 5 (vehicles)
                bbox = results.boxes.xyxy[i].cpu().numpy().tolist()
                conf = float(results.boxes.conf[i].cpu().numpy())
                
                if conf > conf_threshold:
                    boxes.append((bbox, conf, int(cls)))
                    vehicle_boxes.append(bbox)

        # Update DeepSORT tracker
        tracks = tracker.update_tracks(boxes, frame=curr_frame)
        
        # Process movement detection with our advanced detector
        movement_results, camera_motion = movement_detector.process_frame(
            curr_frame, prev_frame, vehicle_boxes, tracks
        )
        
        # Update stats
        stats['total'] = len([t for t in tracks if t.is_confirmed()])
        stats['moving'] = sum(1 for r in movement_results if r['is_moving'])
        stats['static'] = stats['total'] - stats['moving']
        stats['frame'] = frame_count
        stats['front_facing'] = sum(1 for r in movement_results if r.get('front_facing', False))
        stats['camera_approaching'] = sum(1 for r in movement_results if r.get('camera_approaching', False))
        if camera_motion and 'data' in camera_motion and camera_motion['data']:
            stats['camera_motion'] = camera_motion['data'].get('magnitude', 0)
        
        # Add per-frame stats output
        print(f"\nFrame {frame_count}: {stats['moving']} moving, {stats['static']} static vehicles | {stats['front_facing']} front-facing, {stats['camera_approaching']} cam-approach")
        
        # Visualize results
        for result in movement_results:
            track_id = result['track_id']
            bbox = result['bbox']
            is_moving = result['is_moving']
            confidence = result['confidence']
            
            # Get characteristics from result
            depth = result.get('depth', 0)
            front_facing = result.get('front_facing', False)
            in_intersection = result.get('in_intersection', False)
            camera_approaching = result.get('camera_approaching', False)
            
            # Get movement vectors for visualization
            raw_dx = result.get('raw_dx', 0)
            raw_dy = result.get('raw_dy', 0)
            comp_dx = result.get('comp_dx', 0)
            comp_dy = result.get('comp_dy', 0)
            
            # Get trail for visualization
            trail = result.get('trail', None)
            
            # Format label based on characteristics
            label_base = f"Vehicle {track_id}"
            if is_moving:
                state_text = "MOVING"
            else:
                state_text = "STATIC"
                
            # Add descriptors
            descriptors = []
            if depth > movement_detector.far_threshold:
                descriptors.append("far")
            if front_facing:
                descriptors.append("facing")
            if in_intersection:
                descriptors.append("at int.")
            if camera_approaching:
                descriptors.append("cam-app")
                
            # Format label
            if descriptors:
                descriptor_text = "/".join(descriptors)
                label = f"{label_base} - {state_text}({descriptor_text}) - D:{depth:.2f}"
            else:
                label = f"{label_base} - {state_text} - D:{depth:.2f}"
            
            # Draw track visualization
            draw_bounding_box(curr_frame, bbox, label, is_moving, confidence, trail)
            
            # Draw movement vectors
            draw_movement_vector(curr_frame, bbox, raw_dx, raw_dy, comp_dx, comp_dy)
        
        # Add timestamp and stats overlay
        cv2.putText(curr_frame, f"Frame: {frame_count}", (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(curr_frame, f"Vehicles: {stats['total']} | Moving: {stats['moving']} | Static: {stats['static']} | Facing: {stats['front_facing']}", 
                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Add settings info
        cv2.putText(curr_frame, f"Dist Sens: {movement_detector.distance_sensitivity:.1f} | Far Thresh: {movement_detector.far_threshold:.1f} | Boost: {movement_detector.small_movement_boost_factor:.2f}",
                   (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Draw camera motion information
        draw_motion_overlay(curr_frame, camera_motion)
        
        # Write the frame to output video
        out.write(curr_frame)
        
        # Update prev_frame for next iteration
        prev_frame = curr_frame.copy()

    # Clean up
    cap.release()
    out.release()
    print("\n✅ Output saved to output_enhanced.mp4")


if __name__ == "__main__":
    main()

Initialized Enhanced Movement Detector with improved tracking and motion detection
Starting Enhanced Movement Detector with Camera Motion Compensation
Distance sensitivity: 7.0
Distant object boost: 0.4
Far threshold: 0.6
Headlight threshold: 500
Intersection threshold factor: 0.4
Camera motion area factor: 0.007
Small movement boost threshold: 6.0
Small movement boost factor: 0.25
Processing frame 1
0: 256x416 1 crosswalk, 4 pedestrianss, 2 vehicles, 15.9ms
Speed: 2.9ms preprocess, 15.9ms inference, 6.5ms postprocess per image at shape (1, 3, 256, 416)

Frame 1: 0 moving, 0 static vehicles | 0 front-facing, 0 cam-approach
Processing frame 2
0: 256x416 1 crosswalk, 4 pedestrianss, 2 vehicles, 13.5ms
Speed: 2.0ms preprocess, 13.5ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 416)

Frame 2: 0 moving, 2 static vehicles | 0 front-facing, 0 cam-approach
Processing frame 3
0: 256x416 2 crosswalks, 4 pedestrianss, 3 vehicles, 13.2ms
Speed: 2.4ms preprocess, 13.2ms inference, 2